In [2]:
import pickle
import numpy as np
import pandas as pd
import jax.numpy as jnp
import jax
import tensorflow_probability.substrates.jax.bijectors as tfb
from IPython.display import display

# 1. Load the generated samples
filename = "../Data/liesel_output_margarine_paper_redo_40000_samples.pkl" 

with open(filename, "rb") as f:
    res = pickle.load(f)

samples = res["samples"]

# Define Labels
param_names = ["Blue Bonnet", "Fleischmanns", "House", "Generic", "Shed Spread", "LogPrice"]
demo_names = ["Intercept", "log(Income)", "Family size"]

k_dim = len(param_names)

# ==========================================
# REPRODUCING TABLE 5.2: Posterior of Delta
# ==========================================
delta_draws = samples["Delta"].reshape(-1, len(demo_names), k_dim)

delta_mean = np.mean(delta_draws, axis=0)
delta_std = np.std(delta_draws, axis=0)

print("=== TABLE 5.2: Posterior distribution of Delta ===")
delta_df = pd.DataFrame(index=demo_names, columns=param_names)

for i in range(len(demo_names)):
    for j in range(k_dim):
        delta_df.iloc[i, j] = f"{delta_mean[i, j]: .2f} ({delta_std[i, j]:.2f})"

display(delta_df)
print("\n")


# ===============================================
# REPRODUCING TABLE 5.1: Covariance / Correlation
# ===============================================
# Extract latent samples and flatten the chains
# For k=6, the latent vector size is 6 * 7 / 2 = 21
latent_samples = samples["sigma_inv_chol_latent"].reshape(-1, int(k_dim * (k_dim + 1) / 2))

# Setup the bijector (must match the model definition)
bijector = tfb.FillScaleTriL()

# Function to turn 1D vector -> Lower Triangular Matrix -> Covariance Matrix
def latent_to_cov(latent_vec):
    # L is the Cholesky factor of the precision matrix (V_inv)
    L = bijector.forward(latent_vec)
    
    # Precision = L @ L.T
    precision_mat = L @ L.T
    
    # Covariance = inv(Precision)
    return jnp.linalg.inv(precision_mat)

# Vectorize across all draws for fast computation
v_latent_to_cov = jax.vmap(latent_to_cov)
cov_draws = np.array(v_latent_to_cov(latent_samples)) # Convert back to NumPy for easier indexing

n_draws = cov_draws.shape[0]
std_draws = np.zeros((n_draws, k_dim))
corr_draws = np.zeros((n_draws, k_dim, k_dim))

# Transform Covariance to Stds and Correlations per draw
for i in range(n_draws):
    covariance_matrix = cov_draws[i]
    
    # Standard Deviations (sqrt of diagonal)
    stds = np.sqrt(np.diag(covariance_matrix))
    std_draws[i] = stds
    
    # Correlation Matrix = V / (stds * stds^T)
    outer_stds = np.outer(stds, stds)
    corr_draws[i] = covariance_matrix / outer_stds

# Calculate Posterior Means and Standard Deviations
std_mean = np.mean(std_draws, axis=0)
std_sd = np.std(std_draws, axis=0)

corr_mean = np.mean(corr_draws, axis=0)
corr_sd = np.std(corr_draws, axis=0)

print("=== TABLE 5.1: Correlations and standard deviations of betas ===")
table_5_1_df = pd.DataFrame(index=param_names, columns=param_names)

for i in range(k_dim):
    for j in range(k_dim):
        if i == j:
            # Diagonal: Standard Deviations
            table_5_1_df.iloc[i, j] = f"{std_mean[i]:.2f} ({std_sd[i]:.2f})"
        elif j > i:
            # Upper triangle: Correlations
            table_5_1_df.iloc[i, j] = f"{corr_mean[i, j]:.2f} ({corr_sd[i, j]:.2f})"
        else:
            # Lower triangle: leave blank
            table_5_1_df.iloc[i, j] = ""

display(table_5_1_df)

=== TABLE 5.2: Posterior distribution of Delta ===


,Blue Bonnet,Fleischmanns,House,Generic,Shed Spread,LogPrice
Intercept,-1.14 (0.13),-3.63 (0.76),-2.47 (0.19),-5.01 (0.31),-1.76 (0.35),-4.03 (0.17)
log(Income),0.03 (0.23),0.98 (0.72),-0.02 (0.32),-0.64 (0.42),-0.67 (0.45),-0.31 (0.30)
Family size,0.04 (0.30),-2.33 (0.94),0.75 (0.42),2.00 (0.58),0.17 (0.61),0.38 (0.39)




=== TABLE 5.1: Correlations and standard deviations of betas ===


,Blue Bonnet,Fleischmanns,House,Generic,Shed Spread,LogPrice
Blue Bonnet,1.55 (0.13),0.36 (0.14),0.42 (0.10),0.43 (0.10),0.27 (0.13),-0.11 (0.15)
Fleischmanns,,3.97 (0.61),0.24 (0.15),0.11 (0.18),0.07 (0.18),0.37 (0.17)
House,,,2.39 (0.18),0.84 (0.04),0.49 (0.10),-0.13 (0.15)
Generic,,,,2.93 (0.26),0.54 (0.10),-0.25 (0.15)
Shed Spread,,,,,3.02 (0.33),0.06 (0.16)
LogPrice,,,,,,1.43 (0.18)
